# ReservoirLSTM — HO DAK MI 4 (rid=2)
**Kiến trúc**: Hindcast Bi-LSTM + Cross-Attention + NWP Embedding + Quantile Head (7 mức)
**Target**: Dự báo lưu lượng vào hồ `inflow_m3s`
**1 model độc lập cho hồ này** — không train chung với 15 hồ còn lại.

> **Trước khi chạy**: Vào panel **Input** bên phải -> Add Input -> tìm dataset
> chứa `v2_X_hindcast.npy` (build từ `main_build_dataset.py --rid 2` trên máy
> có Data_Tung_Ho_Ma_Tran_Rong/, upload thư mục `datasets/HO_DAK_MI_4/` lên
> làm Kaggle Dataset). Nếu đặt tên dataset khác, sửa `DATA_DIR` ở Cell Setup.


In [ ]:
import os, json, math, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm import tqdm
warnings.filterwarnings("ignore")

RID = 2
RESERVOIR_NAME = 'HO DAK MI 4'
RESERVOIR_KEY  = 'HO_DAK_MI_4'

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Auto-detect: tim thu muc chua v2_X_hindcast.npy
DATA_DIR = None
_input = "/kaggle/input"
_target = "v2_X_hindcast.npy"
for _root, _dirs, _files in os.walk(_input):
    if _target in _files:
        DATA_DIR = _root
        break

if DATA_DIR is None:
    print("Full tree under /kaggle/input:")
    for _root, _dirs, _files in os.walk(_input):
        _depth = _root.replace(_input, "").count(os.sep)
        print("  " * _depth + os.path.basename(_root) + "/", _files[:5])
    raise FileNotFoundError(
        "Khong tim thay dataset. Vao panel Input -> Add Input -> "
        f"attach dataset chua v2_*.npy cua {RESERVOIR_NAME}."
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Reservoir: {RESERVOIR_NAME} (rid={RID})")
print(f"Device   : {device}")
if device.type == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
print(f"DATA_DIR : {DATA_DIR}")
print(f"Files    : {sorted(os.listdir(DATA_DIR))}")


## Config

In [ ]:
"""
Config cho LSTM_Py_Backend_v2 — 1 model độc lập cho MỖI hồ (không reservoir
embedding, không train chung 16 hồ).

Lý do tách khỏi lstm_service/config/settings_v2.py (FloodLSTMv2Config):
  - Baseline NSE cũ (artifacts/plots/metrics.txt, lstm_service) cho thấy model
    chung có độ lệch rất lớn giữa các hồ (0.87 xuống tới âm) — nghi ngờ 1 vài
    hồ dữ liệu xấu (xem INFLOW_CAPS_M3S bên dưới, đánh dấu "lỗi data") kéo NSE
    trung bình xuống hoặc gây nhiễu học chung.
  - Train riêng từng hồ: đổi 1 hồ dữ liệu xấu KHÔNG ảnh hưởng tới model của
    các hồ khác, dễ debug/so sánh NSE per-reservoir hơn.

Mặc định hindcast/forecast GIỮ NGUYÊN theo hợp đồng production hiện tại
(SEQ_LENGTH=240h/10 ngày, HORIZON=24h — xem lstm_service/config/settings.py)
để model mới có thể thay thế v1 sau này mà không cần đổi phía Node.js/cron.
Có thể chỉnh forecast_len=168 (7 ngày, kiểu Google FloodHub) nếu muốn.
"""

from dataclasses import dataclass, field


@dataclass
class ReservoirLSTMConfig:
    # ── Hồ đang train (bắt buộc set trước khi build dataset/train) ─────────────
    rid: int = 0                     # key trong config/reservoirs.py RESERVOIRS
    reservoir_name: str = ""         # điền tự động từ RESERVOIRS[rid]["name"]

    # ── Sequence lengths (giữ theo hợp đồng production v1) ──────────────────────
    hindcast_len: int = 240          # 10 ngày lịch sử (hourly) — SEQ_LENGTH cũ
    forecast_len: int = 24           # 24h dự báo — HORIZON cũ
    #   Muốn thử 7-ngày kiểu FloodLSTMv2/Google: hindcast_len=720, forecast_len=168

    # ── Model dimensions ──────────────────────────────────────────────────────
    hidden_size: int = 128           # nhỏ hơn bản dùng chung (256) vì data/model giờ nhỏ hơn nhiều
    num_layers: int = 2
    nwp_embed_dim: int = 32

    # ── Input features ─────────────────────────────────────────────────────────
    # 47 = 18 rain + 12 inflow + 6 reservoir + 5 meteo (temp/rh/pressure/et0/wind,
    # đủ 5 vì giờ có Open-Meteo archive — xem data/nwp_fetcher.py) + 6 thời gian
    n_hindcast_features: int = 47    # xem data/dataset_builder.py FEATURES
    n_nwp_features: int = 6          # rain_fc, rain_fc_3h, rain_fc_6h, rain_fc_24h, temp_fc, wind_fc
    n_nwp_sources: int = 1           # chỉ Open-Meteo — không có nguồn dự phòng như bản chung

    # ── Học trọng số trạm mưa (tùy chọn, TẮT mặc định) ───────────────────────────
    # models/station_attention.py đã sẵn sàng nhưng dataset_builder.py hiện CHƯA
    # trích xuất mưa từng trạm riêng lẻ từ Data_Tung_Ho_Ma_Tran_Rong/*.xlsx (file
    # có cột "Col 121+: Dữ liệu trạm riêng lẻ" nhưng rain_matrix_loader.py hiện
    # bỏ qua — cần xem cấu trúc cột thật trước khi parse, xem TODO trong
    # data/dataset_builder.py). Bật cờ này CHỈ sau khi có station_rain/station_mask.
    use_station_attention: bool = False
    max_stations: int = 7            # số trạm tối đa/hồ, xem RESERVOIR_TO_STATIONS

    # ── Quantile output (7 mức, giống FloodLSTMv2) ───────────────────────────────
    quantiles: list = field(
        default_factory=lambda: [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )

    # ── Training hyperparameters ────────────────────────────────────────────────
    batch_size: int = 128
    epochs: int = 100
    lr: float = 3e-4
    weight_decay: float = 1e-3
    warmup_epochs: int = 10
    grad_clip: float = 1.0
    patience: int = 20

    teacher_forcing_start: float = 0.8
    teacher_forcing_end: float = 0.0

    oversample_p95_factor: int = 2
    oversample_p99_factor: int = 3

    target_noise_std: float = 0.005

    # ── Training splits (fixed-date, giữ theo train_global.py) ─────────────────
    train_end: str = "2024-08-31"
    val_start: str = "2024-09-01"
    val_end: str = "2025-01-01"
    test_start: str = "2025-09-01"

    # ── Paths ──────────────────────────────────────────────────────────────────
    data_dir: str = "."
    artifacts_dir: str = "artifacts"

    @property
    def n_quantiles(self) -> int:
        return len(self.quantiles)

    @property
    def median_idx(self) -> int:
        return self.n_quantiles // 2

    def teacher_forcing_ratio(self, epoch: int) -> float:
        p = epoch / max(self.epochs - 1, 1)
        return self.teacher_forcing_start * (1 - p) + self.teacher_forcing_end * p


# ═══════════════════════════════════════════════════════════════════════════════
# Giới hạn inflow hợp lý (m3/s) — copy từ lstm_service/data/dataset_builder.py.
# Các hồ đánh dấu "lỗi data" là nghi phạm hàng đầu cho NSE thấp trong baseline
# cũ (Song Bung 2 idx6, Song Bung 6 idx7, Dak Mi 3 idx10, Khe Dien idx11,
# Dak Mi 2 idx14) — ưu tiên kiểm tra lại coverage/outlier khi train hồ này.
# ═══════════════════════════════════════════════════════════════════════════════
INFLOW_CAPS_M3S = {
    1:  2500,   # HO A VUONG        (idx=0)
    2:  4500,   # HO DAK MI 4       (idx=1)
    3:  4500,   # HO SONG BUNG 4    (idx=2)
    4:  8000,   # HO SONG TRANH 2   (idx=3)
    7:  7000,   # HO SONG BUNG 4A   (idx=4)
    8:  7000,   # HO SONG BUNG 5    (idx=5)
    9:  800,    # HO SONG BUNG 2    (idx=6)  — lỗi data (p99.9=617, max quan trắc=19701)
    11: 8000,   # HO SONG BUNG 6    (idx=7)  — lỗi data (p99.9=5946, max quan trắc=40232)
    12: 12000,  # HO SONG TRANH 3   (idx=8)
    13: 2500,   # HO ZA HUNG        (idx=9)
    14: 2000,   # HO DAK MI 3       (idx=10) — lỗi data (p99.9=1350, max quan trắc=35321)
    15: 1000,   # HO KHE DIEN       (idx=11) — lỗi data (p99.9=729,  max quan trắc=6087)
    16: 900,    # HO SONG CON 2     (idx=12)
    17: 13000,  # HO SONG TRANH 4   (idx=13)
    18: 2500,   # HO DAK MI 2       (idx=14) — lỗi data rõ ràng (p99.9=1537, max quan trắc=391526)
    19: 700,    # HO DAK MI 4C      (idx=15)
}


In [ ]:
CFG = ReservoirLSTMConfig(rid=RID, reservoir_name=RESERVOIR_NAME)
CFG.artifacts_dir = OUTPUT_DIR
print(CFG)


## Model — StationRainAttention (tùy chọn, tắt mặc định)

In [ ]:
# models/station_attention.py
"""
StationRainAttention — bản 1-hồ (không có reservoir embedding/context).

Khác bản ở lstm_service/models/station_attention.py: bản đó phục vụ 1 model
CHUNG nhiều hồ nên cần bảng bias theo (reservoir, station) + reservoir embedding
làm context cho score_net. Ở đây mỗi model chỉ phục vụ ĐÚNG 1 hồ, nên:
  - prior_bias là 1 vector duy nhất (không phải bảng tra theo hồ)
  - score_net chỉ nhận giá trị mưa hiện tại (không cần context "hồ nào")

Thiết kế attention giữ nguyên ý tưởng gốc (lấy cảm hứng từ AttenCLSTM trong
CNN-LSTM-Attention-Model-for-Runoff-Prediction): giữ IDW làm prior vật lý
(bias khởi tạo từ log(idw_weight) qua init_prior()), rồi để attention học điều
chỉnh dần trong quá trình train. Trạm thiếu dữ liệu tại thời điểm t bị loại
khỏi softmax qua mask.

TẮT mặc định (config.use_station_attention=False) — xem TODO trong
data/dataset_builder.py về việc parse mưa từng trạm từ Excel.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class StationRainAttention(nn.Module):
    """
    Input:
        station_rain : (B, T, S) — mưa thô từng trạm (mm), 0 nếu thiếu
        station_mask : (B, T, S) bool — True nếu trạm hợp lệ tại t

    Output:
        learned_rain : (B, T, 1) — mưa lưu vực đã học trọng số
        attn_weights : (B, T, S) — trọng số attention (để log/debug)
    """

    def __init__(self, max_stations: int, hidden_dim: int = 16):
        super().__init__()
        self.max_stations = max_stations

        # Bias tĩnh theo trạm — khởi tạo từ trọng số IDW (init_prior), học tiếp khi train
        self.prior_bias = nn.Parameter(torch.zeros(max_stations))

        # Logit động theo giá trị mưa hiện tại tại mỗi trạm
        self.score_net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.xavier_uniform_(self.score_net[0].weight)
        nn.init.zeros_(self.score_net[0].bias)
        nn.init.zeros_(self.score_net[2].weight)
        nn.init.zeros_(self.score_net[2].bias)

    def init_prior(self, weights: list):
        """Khởi tạo bias từ trọng số IDW tĩnh đã normalize (sum=1), log space."""
        padded = list(weights) + [0.0] * (self.max_stations - len(weights))
        log_w = [math.log(w) if w > 1e-8 else -20.0 for w in padded[: self.max_stations]]
        with torch.no_grad():
            self.prior_bias.copy_(torch.tensor(log_w, dtype=torch.float32))

    def forward(self, station_rain: torch.Tensor, station_mask: torch.Tensor):
        B, T, S = station_rain.shape
        assert S == self.max_stations, f"expected {self.max_stations} stations, got {S}"

        bias = self.prior_bias.view(1, 1, -1).expand(B, T, -1)   # (B, T, S)
        dyn_score = self.score_net(station_rain.unsqueeze(-1)).squeeze(-1)  # (B, T, S)

        logits = bias + dyn_score
        logits = logits.masked_fill(~station_mask, float("-inf"))

        no_valid = (~station_mask).all(dim=-1, keepdim=True)      # (B, T, 1)
        safe_logits = torch.where(no_valid.expand_as(logits), torch.zeros_like(logits), logits)

        attn = F.softmax(safe_logits, dim=-1)
        attn = attn.masked_fill(no_valid.expand_as(attn), 0.0)

        learned_rain = (attn * station_rain).sum(dim=-1, keepdim=True)  # (B, T, 1)
        return learned_rain, attn


## Model — ReservoirLSTM

In [ ]:
# models/flood_lstm_v2.py
"""
ReservoirLSTM — bản 1-hồ của FloodLSTM v2 (lstm_service/models/flood_lstm_v2.py).

Khác bản gốc:
  1. KHÔNG có reservoir embedding / reservoir_idx — mỗi model chỉ phục vụ 1 hồ
     (xem config/settings.py ReservoirLSTMConfig, context ở đây được lấy hết
     từ chính dữ liệu của hồ đó, không cần phân biệt "hồ nào").
  2. NWPEmbedding thay NWPFusionLayer — bản gốc hỗ trợ nhiều nguồn NWP với
     attention theo availability mask; ở đây chỉ có 1 nguồn (Open-Meteo) nên
     đơn giản hoá thành 1 projection layer.
  3. StationRainAttention (tùy chọn) không cần reservoir context nữa — xem
     models/station_attention.py bản 1-hồ.

Giữ nguyên từ bản gốc (không phụ thuộc số hồ):
  - Hindcast Encoder (Bi-LSTM) + Forecast Decoder (LSTM autoregressive)
  - Cross-attention: decoder query -> hindcast encoder key/value
  - Horizon-aware uncertainty scaling (P50 cố định, P5/P95 giãn theo lead-time)
  - Quantile head 7 mức
"""

import torch
import torch.nn as nn
import torch.nn.functional as F




# ═══════════════════════════════════════════════════════════════════════════════
# NWP Embedding — đơn giản hoá NWPFusionLayer (bản gốc hỗ trợ multi-source)
# ═══════════════════════════════════════════════════════════════════════════════

class NWPEmbedding(nn.Module):
    """Project NWP features (rain/temp/wind...) -> embedding, 1 nguồn duy nhất."""

    def __init__(self, input_dim: int, embed_dim: int):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # (B, T, input_dim) -> (B, T, embed_dim)
        return self.proj(x)


# ═══════════════════════════════════════════════════════════════════════════════
# Hindcast Cross-Attention
# ═══════════════════════════════════════════════════════════════════════════════

class HindcastCrossAttention(nn.Module):
    """
    Cross-attention: forecast decoder query -> hindcast encoder key/value.
    Cho phép decoder tập trung vào các thời điểm quan trọng trong lịch sử
    (vd: đỉnh lũ trước đó, trạng thái mưa dài hạn).
    """

    def __init__(self, hidden_size: int, n_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=n_heads, dropout=0.1, batch_first=True,
        )
        self.norm = nn.LayerNorm(hidden_size)

    def forward(self, query: torch.Tensor, key_value: torch.Tensor) -> torch.Tensor:
        ctx, _ = self.attn(query, key_value, key_value)
        out = self.norm(ctx + query)
        return out.squeeze(1)  # (B, H)


# ═══════════════════════════════════════════════════════════════════════════════
# ReservoirLSTM — Main Model
# ═══════════════════════════════════════════════════════════════════════════════

class ReservoirLSTM(nn.Module):
    """
    Two-phase flood forecasting model cho 1 hồ.

    Phase 1 — Hindcast:
        Input : x_hindcast (B, hindcast_len, n_hindcast_features)
        Encoder: Bi-LSTM 2 layers -> final state -> project -> decoder init
        Encoder output: (B, hindcast_len, H) -> key/value cho cross-attention

    Phase 2 — Forecast (autoregressive):
        NWP input: (B, forecast_len, n_nwp_features) — Open-Meteo, 1 nguồn
        Decoder: LSTM 2 layers + cross-attention -> 7 quantiles mỗi bước

    Output: (B, forecast_len, 7) trong sqrt(Q) space, đã sort monotonic
    """

    def __init__(self, config):
        super().__init__()
        H  = config.hidden_size
        NQ = config.n_quantiles

        # ── Học trọng số trạm mưa (tùy chọn) ──────────────────────────────────
        self.use_station_attention = getattr(config, "use_station_attention", False)
        if self.use_station_attention:
            self.station_attn = StationRainAttention(max_stations=config.max_stations)
        station_extra = 1 if self.use_station_attention else 0

        # ── Phase 1: Hindcast Encoder ──────────────────────────────────────────
        self.hindcast_proj = nn.Sequential(
            nn.Linear(config.n_hindcast_features + station_extra, H),
            nn.LayerNorm(H),
            nn.GELU(),
        )
        self.hindcast_encoder = nn.LSTM(
            input_size=H, hidden_size=H, num_layers=config.num_layers,
            dropout=0.2 if config.num_layers > 1 else 0.0,
            bidirectional=True, batch_first=True,
        )
        self.enc_kv_proj   = nn.Linear(H * 2, H)
        self.h_state_proj  = nn.Linear(H * 2, H)
        self.c_state_proj  = nn.Linear(H * 2, H)

        # ── Phase 2a: NWP Embedding ─────────────────────────────────────────────
        self.nwp_embed = NWPEmbedding(config.n_nwp_features, config.nwp_embed_dim)

        # ── Phase 2b: Cross-Attention (decoder -> hindcast) ─────────────────────
        self.cross_attn = HindcastCrossAttention(H, n_heads=4)

        # ── Phase 2c: Forecast Decoder ───────────────────────────────────────────
        dec_input_dim = config.nwp_embed_dim + NQ + H
        self.forecast_decoder = nn.LSTM(
            input_size=dec_input_dim, hidden_size=H, num_layers=config.num_layers,
            dropout=0.2 if config.num_layers > 1 else 0.0, batch_first=True,
        )

        # ── Output head ────────────────────────────────────────────────────────
        self.output_head = nn.Sequential(
            nn.Linear(H, H // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(H // 2, NQ),
        )

        # Horizon uncertainty scale: bước sau -> khoảng tin cậy rộng hơn (học được)
        self.horizon_unc_scale = nn.Parameter(
            torch.linspace(1.0, 2.5, config.forecast_len).unsqueeze(1)
        )  # (forecast_len, 1)

        self.config = config
        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if "weight_ih" in name or "weight_hh" in name:
                nn.init.orthogonal_(p)
            elif "bias" in name and p.dim() == 1 and "horizon_unc_scale" not in name:
                nn.init.zeros_(p)

    def _project_encoder_state(self, h_n: torch.Tensor, c_n: torch.Tensor):
        """Gộp forward+backward directions -> decoder initial state."""
        n_layers = self.config.num_layers
        h_fwd, h_bwd = h_n[0::2], h_n[1::2]
        c_fwd, c_bwd = c_n[0::2], c_n[1::2]
        h_cat = torch.cat([h_fwd, h_bwd], dim=-1)
        c_cat = torch.cat([c_fwd, c_bwd], dim=-1)
        h0 = self.h_state_proj(h_cat).contiguous()
        c0 = self.c_state_proj(c_cat).contiguous()
        return h0, c0

    def forward(
        self,
        x_hindcast: torch.Tensor,          # (B, hindcast_len, n_hindcast_features)
        x_nwp: torch.Tensor,               # (B, forecast_len, n_nwp_features)
        teacher_forcing_ratio: float = 0.0,
        y_true_sqrt: torch.Tensor = None,  # (B, forecast_len) ground truth, sqrt space
        station_rain: torch.Tensor = None, # (B, hindcast_len, max_stations)
        station_mask: torch.Tensor = None, # (B, hindcast_len, max_stations) bool
    ) -> torch.Tensor:                     # (B, forecast_len, n_quantiles)

        B = x_hindcast.size(0)
        device = x_hindcast.device
        NQ  = self.config.n_quantiles
        med = self.config.median_idx

        # ── Phase 1: Hindcast Encoding ──────────────────────────────────────────
        if self.use_station_attention and station_rain is not None:
            learned_rain, _ = self.station_attn(station_rain, station_mask)
            enc_input_raw = torch.cat([x_hindcast, learned_rain], dim=-1)
        else:
            enc_input_raw = x_hindcast
        enc_input = self.hindcast_proj(enc_input_raw)

        enc_out, (h_n, c_n) = self.hindcast_encoder(enc_input)   # enc_out: (B, T_h, 2H)
        enc_kv = self.enc_kv_proj(enc_out)                       # (B, T_h, H)
        h0, c0 = self._project_encoder_state(h_n, c_n)

        # ── Phase 2a: NWP Embedding ──────────────────────────────────────────────
        nwp_emb = self.nwp_embed(x_nwp)   # (B, forecast_len, nwp_embed_dim)

        # ── Phase 2b+c: Autoregressive Forecast ─────────────────────────────────
        outputs = []
        prev_q = torch.zeros(B, NQ, device=device)
        h_dec, c_dec = h0, c0

        for t in range(self.config.forecast_len):
            query = h_dec[-1].unsqueeze(1)          # (B, 1, H)
            ctx = self.cross_attn(query, enc_kv)     # (B, H)

            nwp_t = nwp_emb[:, t, :]                 # (B, nwp_embed_dim)
            dec_in = torch.cat([nwp_t, prev_q, ctx], dim=-1).unsqueeze(1)

            dec_out, (h_dec, c_dec) = self.forecast_decoder(dec_in, (h_dec, c_dec))
            raw_q = self.output_head(dec_out.squeeze(1))  # (B, NQ)

            scale = self.horizon_unc_scale[t].to(device)
            median_pred = raw_q[:, med:med + 1]
            scaled_q = median_pred + (raw_q - median_pred) * scale

            outputs.append(scaled_q)

            if teacher_forcing_ratio > 0.0 and y_true_sqrt is not None:
                use_gt = torch.rand(B, device=device) < teacher_forcing_ratio
                gt_q = prev_q.clone()
                gt_q[:, med] = y_true_sqrt[:, t]
                prev_q = torch.where(use_gt.unsqueeze(-1).expand_as(scaled_q), gt_q, scaled_q.detach())
            else:
                prev_q = scaled_q.detach()

        preds = torch.stack(outputs, dim=1)
        preds = torch.sort(preds, dim=-1).values
        return preds


## Loss — Quantile + Peak-aware + Coverage

In [ ]:
# models/quantile_loss_v2.py
"""
Loss function cho ReservoirLSTM — copy từ lstm_service/models/quantile_loss_v2.py
(logic không phụ thuộc số hồ, giữ nguyên).

Thành phần:
  1. Weighted Pinball Loss  — quantile regression chuẩn, 7 quantiles
  2. Peak-Aware MSE on P50  — ưu tiên đỉnh lũ, tránh mode collapse
  3. Horizon Decay          — bước gần hơn được weight cao hơn
  4. Coverage Bonus         — khuyến khích P5-P95 bao phủ thực tế
"""

import torch
import torch.nn.functional as F


def quantile_loss_v2(
    preds: torch.Tensor,        # (B, T, NQ) — sqrt space, sorted
    targets: torch.Tensor,      # (B, T)     — sqrt space
    quantiles: list,
    horizon_decay: float = 0.02,     # decay nhanh hơn bản 168h vì forecast_len ngắn (24h mặc định)
    coverage_weight: float = 0.05,
    peak_weight: float = 0.15,
) -> torch.Tensor:
    device = preds.device
    B, T, NQ = preds.shape
    med_idx = len(quantiles) // 2

    qs = torch.tensor(quantiles, dtype=torch.float32, device=device)

    # ── 1. Pinball (Quantile) Loss ─────────────────────────────────────────────
    err = targets.unsqueeze(-1) - preds
    pinball = torch.max(qs * err, (qs - 1.0) * err)

    p50_bias = torch.ones(NQ, device=device)
    p50_bias[med_idx] = 1.1
    pinball = pinball * p50_bias.view(1, 1, -1)

    # ── 2. Horizon Decay ───────────────────────────────────────────────────────
    t_weights = torch.exp(-horizon_decay * torch.arange(T, dtype=torch.float32, device=device))
    t_weights = t_weights / t_weights.sum()

    pinball_loss = (pinball * t_weights.view(1, -1, 1)).sum(dim=1).mean()

    # ── 3. Peak-Aware MSE on P50 ───────────────────────────────────────────────
    median_pred = preds[:, :, med_idx]
    peak_w = torch.sqrt(targets + 1.0)
    peak_w = peak_w / (peak_w.mean() + 1e-8)
    peak_w = peak_w.clamp(max=5.0)

    mse_peak = (peak_w * (median_pred - targets) ** 2)
    mse_peak = (mse_peak * t_weights.view(1, -1)).sum(dim=1).mean()

    # ── 4. Coverage Bonus ─────────────────────────────────────────────────────
    p_low  = preds[:, :, 0]
    p_high = preds[:, :, -1]
    below = F.relu(p_low  - targets)
    above = F.relu(targets - p_high)
    coverage_loss = (
        (below * t_weights.view(1, -1)).sum(dim=1).mean() +
        (above * t_weights.view(1, -1)).sum(dim=1).mean()
    )

    total = (
        (1.0 - peak_weight - coverage_weight) * pinball_loss
        + peak_weight * mse_peak
        + coverage_weight * coverage_loss
    )
    return total


## Metrics — NSE theo lead-time + chẩn đoán từng trận lũ

In [ ]:
# training/event_metrics.py
"""
Copy nguyên từ lstm_service/training/event_metrics.py — logic không phụ thuộc
số hồ, dùng lại y hệt cho bản 1-hồ.

Chẩn đoán bổ sung cho đánh giá test — lấy cảm hứng từ get_peak() trong
repo tham khảo CNN-LSTM-Attention-Model-for-Runoff-Prediction
(aba-hash/CNN-LSTM-Attention-Model-for-Runoff-Prediction, data_processing.py).

  1. nse_per_horizon()      — NSE riêng cho từng nhóm lead-time.
  2. detect_flood_events() + flood_event_diagnostics()
                             — tách từng trận lũ riêng lẻ (rise -> peak -> fall)
                               thay vì NSE gộp trên top-N% timestep, tính NSE và
                               sai số đỉnh (relative error) cho TỪNG trận.
"""

import numpy as np


def nse_single(obs: np.ndarray, pred: np.ndarray) -> float:
    """NSE (Nash-Sutcliffe Efficiency) trên 1 mảng 1D đã flatten."""
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    if ss_tot < 1e-8:
        return float("nan")
    return 1.0 - ss_res / ss_tot


def extract_lead_time_series(
    preds: np.ndarray,   # (N, T) point forecast, đơn vị gốc (m3/s)
    obs: np.ndarray,     # (N, T)
    lead_idx: int,       # 0-based: 0 = giờ thứ 1, 23 = giờ thứ 24, ...
):
    """
    Trích chuỗi liên tục obs/pred tại 1 lead-time cố định từ tập test dạng
    sliding-window. Giả định val/test loader shuffle=False.
    """
    return obs[:, lead_idx].copy(), preds[:, lead_idx].copy()


def nse_per_horizon(
    preds: np.ndarray,   # (N, T) point forecast (đơn vị gốc, không phải sqrt)
    obs: np.ndarray,     # (N, T)
    group_hours: int = 6,
) -> list:
    """NSE riêng cho từng nhóm lead-time (mặc định 6h/nhóm cho horizon 24h)."""
    N, T = preds.shape
    n_groups = (T + group_hours - 1) // group_hours
    results = []
    for g in range(n_groups):
        lo, hi = g * group_hours, min((g + 1) * group_hours, T)
        p = preds[:, lo:hi].reshape(-1)
        o = obs[:, lo:hi].reshape(-1)
        results.append({
            "group": g + 1,
            "hour_range": f"{lo + 1}-{hi}h",
            "nse": round(nse_single(o, p), 4),
            "n": int(p.size),
        })
    return results


def detect_flood_events(
    obs: np.ndarray,          # (T,) chuỗi quan trắc liên tục (1 lead-time cố định)
    threshold: float,
    min_separation: int = 24,
) -> list:
    """Tách các trận lũ riêng lẻ khỏi 1 chuỗi quan trắc liên tục."""
    T = len(obs)
    candidate = np.where(obs >= threshold)[0]
    if len(candidate) == 0:
        return []

    peak_indices = []
    i = 0
    while i < len(candidate):
        j = i
        while j + 1 < len(candidate) and candidate[j + 1] - candidate[j] <= min_separation:
            j += 1
        segment = candidate[i:j + 1]
        peak_indices.append(int(segment[np.argmax(obs[segment])]))
        i = j + 1

    events = []
    for peak_idx in peak_indices:
        start = peak_idx
        while start > 0 and obs[start - 1] <= obs[start]:
            start -= 1
        end = peak_idx
        while end + 1 < T and obs[end + 1] <= obs[end]:
            end += 1
        events.append({"start": start, "peak": peak_idx, "end": end})
    return events


def flood_event_diagnostics(
    obs: np.ndarray,
    pred: np.ndarray,
    threshold: float,
    min_separation: int = 24,
    peak_re_tolerance: float = 0.2,
) -> dict:
    """Chẩn đoán từng trận lũ riêng lẻ (NSE + sai số đỉnh + QA pass rate)."""
    events = detect_flood_events(obs, threshold, min_separation)
    if not events:
        return {
            "n_events": 0, "mean_event_nse": float("nan"),
            "peak_re_mean": float("nan"), "qa_pass_rate": float("nan"),
            "events": [],
        }

    details = []
    for ev in events:
        s, p, e = ev["start"], ev["peak"], ev["end"]
        o_seg = obs[s:e + 1]
        p_seg = pred[s:e + 1]
        ev_nse = nse_single(o_seg, p_seg)

        obs_peak = float(obs[p])
        pred_peak_in_window = float(p_seg.max()) if len(p_seg) else float("nan")
        re = abs(pred_peak_in_window - obs_peak) / max(obs_peak, 1e-6)

        details.append({
            "start": s, "peak": p, "end": e,
            "obs_peak": round(obs_peak, 2),
            "pred_peak": round(pred_peak_in_window, 2),
            "peak_re": round(re, 4),
            "event_nse": round(ev_nse, 4) if not np.isnan(ev_nse) else None,
            "qa_pass": bool(re < peak_re_tolerance),
        })

    valid_nse = [d["event_nse"] for d in details if d["event_nse"] is not None]
    return {
        "n_events": len(details),
        "mean_event_nse": round(float(np.mean(valid_nse)), 4) if valid_nse else float("nan"),
        "peak_re_mean": round(float(np.mean([d["peak_re"] for d in details])), 4),
        "qa_pass_rate": round(float(np.mean([d["qa_pass"] for d in details])), 4),
        "events": details,
    }


## Dataset

In [ ]:
# data/reservoir_dataset.py
"""PyTorch Dataset cho 1 hồ — load v2_*.npy do data/dataset_builder.py sinh ra."""

import os
import numpy as np
import torch
from torch.utils.data import Dataset


class ReservoirDataset(Dataset):
    """
    Load datasets/<reservoir_key>/v2_*.npy.

    Mỗi item: (x_hindcast, x_nwp, y, station_rain, station_mask)
      x_hindcast   : (hindcast_len, n_hindcast_features) float32
      x_nwp        : (forecast_len, n_nwp_features)       float32
      y            : (forecast_len,)                       float32 — sqrt(inflow)
      station_rain : (hindcast_len, max_stations)          float32 — 0 nếu chưa build
      station_mask : (hindcast_len, max_stations)          bool    — False nếu chưa build

    station_rain/station_mask CHỈ có ý nghĩa khi config.use_station_attention=True
    (xem models/flood_lstm_v2.py). Luôn trả về đủ 5 phần tử để vòng lặp train/val/
    test không cần if/else riêng.
    """

    def __init__(self, data_dir: str, inflow_cap_sqrt: float | None = None, max_stations: int = 7):
        self.X_hind = np.load(os.path.join(data_dir, "v2_X_hindcast.npy"), mmap_mode="r")
        self.X_nwp  = np.load(os.path.join(data_dir, "v2_X_nwp.npy"),      mmap_mode="r")
        self.y      = np.load(os.path.join(data_dir, "v2_y.npy"),           mmap_mode="r")

        ts_path = os.path.join(data_dir, "v2_timestamps.npy")
        self.timestamps = np.load(ts_path) if os.path.exists(ts_path) else None

        rain_path = os.path.join(data_dir, "v2_station_rain.npy")
        mask_path = os.path.join(data_dir, "v2_station_mask.npy")
        self.has_station_data = os.path.exists(rain_path) and os.path.exists(mask_path)
        if self.has_station_data:
            self.station_rain = np.load(rain_path, mmap_mode="r")
            self.station_mask = np.load(mask_path, mmap_mode="r")
            self.max_stations = self.station_rain.shape[-1]
        else:
            self.station_rain = None
            self.station_mask = None
            self.max_stations = max_stations

        self.inflow_cap_sqrt = inflow_cap_sqrt

        print(
            f"ReservoirDataset({data_dir}): {len(self):,} samples | "
            f"hindcast={self.X_hind.shape[1]}h | forecast={self.y.shape[1]}h | "
            f"station_attention_data={'ON' if self.has_station_data else 'OFF'}"
        )

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        y = np.array(self.y[idx], copy=True, dtype=np.float32)
        if self.inflow_cap_sqrt is not None:
            y = np.clip(y, 0.0, self.inflow_cap_sqrt)

        x_hind = torch.from_numpy(np.array(self.X_hind[idx], copy=False)).float()
        x_nwp  = torch.from_numpy(np.array(self.X_nwp[idx],  copy=False)).float()
        y_t    = torch.from_numpy(y).float()

        if self.has_station_data:
            station_rain = torch.from_numpy(np.array(self.station_rain[idx], copy=False)).float()
            station_mask = torch.from_numpy(np.array(self.station_mask[idx], copy=False)).bool()
        else:
            T_h = x_hind.shape[0]
            station_rain = torch.zeros(T_h, self.max_stations, dtype=torch.float32)
            station_mask = torch.zeros(T_h, self.max_stations, dtype=torch.bool)

        return x_hind, x_nwp, y_t, station_rain, station_mask


In [ ]:
dataset = ReservoirDataset(
    DATA_DIR,
    inflow_cap_sqrt=(math.sqrt(INFLOW_CAPS_M3S[RID]) if RID in INFLOW_CAPS_M3S else None),
    max_stations=CFG.max_stations,
)
print(f"Total samples: {len(dataset):,}")


## Split + Flood Oversampling

In [ ]:
ts = dataset.timestamps
train_end  = np.datetime64(CFG.train_end, "s")
val_start  = np.datetime64(CFG.val_start, "s")
val_end    = np.datetime64(CFG.val_end, "s")
test_start = np.datetime64(CFG.test_start, "s")

all_idx = np.arange(len(ts))
train_idx = all_idx[ts < val_start].tolist()
val_idx   = all_idx[(ts >= val_start) & (ts < val_end)].tolist()
test_idx  = all_idx[ts >= test_start].tolist()
print(f"Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}")

train_y_max = dataset.y[train_idx].max(axis=1)
thr95 = float(np.percentile(train_y_max, 95))
thr99 = float(np.percentile(train_y_max, 99))
idx95 = [train_idx[i] for i in np.where(train_y_max >= thr95)[0]]
idx99 = [train_idx[i] for i in np.where(train_y_max >= thr99)[0]]
oversampled = train_idx + idx95 * CFG.oversample_p95_factor + idx99 * CFG.oversample_p99_factor
print(f"Oversampling: top5%={len(idx95):,}x{CFG.oversample_p95_factor} | "
      f"top1%={len(idx99):,}x{CFG.oversample_p99_factor} | total={len(oversampled):,}")

n_w = 2 if device.type == "cuda" else 0
_kw = dict(num_workers=n_w, pin_memory=(device.type == "cuda"), persistent_workers=(n_w > 0))
train_loader = DataLoader(Subset(dataset, oversampled), batch_size=CFG.batch_size, shuffle=True, **_kw)
val_loader   = DataLoader(Subset(dataset, val_idx),    batch_size=CFG.batch_size, shuffle=False, **_kw)
test_loader  = DataLoader(Subset(dataset, test_idx),   batch_size=CFG.batch_size, shuffle=False, num_workers=n_w)


## Seed + Metrics Helper

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


def nse_np(obs, pred):
    ss_res = float(np.sum((obs - pred) ** 2))
    ss_tot = float(np.sum((obs - obs.mean()) ** 2))
    return float("nan") if ss_tot < 1e-8 else 1.0 - ss_res / ss_tot


def compute_metrics(preds, targets, med_idx):
    pred_med = preds[:, :, med_idx]
    pred_raw   = (pred_med ** 2).numpy()
    target_raw = (targets ** 2).numpy()
    mae  = float(np.mean(np.abs(pred_raw - target_raw)))
    rmse = float(np.sqrt(np.mean((pred_raw - target_raw) ** 2)))
    nse  = nse_np(target_raw.flatten(), pred_raw.flatten())
    return {"mae": mae, "rmse": rmse, "nse": nse}


print("Helpers OK")


## Training Loop

In [ ]:
model = ReservoirLSTM(CFG).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ReservoirLSTM parameters: {n_params:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)

def lr_lambda(epoch):
    if epoch < CFG.warmup_epochs:
        return float(epoch + 1) / CFG.warmup_epochs
    progress = (epoch - CFG.warmup_epochs) / max(CFG.epochs - CFG.warmup_epochs, 1)
    return max(0.05, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
use_amp = device.type == "cuda"
amp_scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"AMP: {'ON' if use_amp else 'OFF'}")

best_val = float("inf")
patience_cnt = 0
history = []
ckpt_path = os.path.join(OUTPUT_DIR, "reservoir_lstm.pt")

for epoch in range(CFG.epochs):
    tf_ratio = CFG.teacher_forcing_ratio(epoch)
    model.train()
    train_loss = 0.0
    for x_hind, x_nwp, y_b, station_rain, station_mask in tqdm(
        train_loader, desc=f"Epoch {epoch+1}/{CFG.epochs}", leave=False
    ):
        x_hind, x_nwp, y_b = x_hind.to(device), x_nwp.to(device), y_b.to(device)
        station_rain, station_mask = station_rain.to(device), station_mask.to(device)
        y_noisy = y_b + torch.randn_like(y_b) * CFG.target_noise_std if CFG.target_noise_std > 0 else y_b

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=use_amp):
            preds = model(x_hind, x_nwp, teacher_forcing_ratio=tf_ratio, y_true_sqrt=y_noisy,
                          station_rain=station_rain, station_mask=station_mask)
            loss = quantile_loss_v2(preds, y_noisy, CFG.quantiles)

        amp_scaler.scale(loss).backward()
        amp_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        amp_scaler.step(optimizer)
        amp_scaler.update()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_hind, x_nwp, y_b, station_rain, station_mask in val_loader:
            x_hind, x_nwp, y_b = x_hind.to(device), x_nwp.to(device), y_b.to(device)
            station_rain, station_mask = station_rain.to(device), station_mask.to(device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                preds = model(x_hind, x_nwp, teacher_forcing_ratio=0.0,
                              station_rain=station_rain, station_mask=station_mask)
                val_loss += quantile_loss_v2(preds, y_b, CFG.quantiles).item()
            all_preds.append(preds.cpu()); all_targets.append(y_b.cpu())

    val_loss /= len(val_loader)
    preds_cat, targets_cat = torch.cat(all_preds), torch.cat(all_targets)
    m = compute_metrics(preds_cat, targets_cat, CFG.median_idx)
    scheduler.step()
    lr_now = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch+1:3d} | LR {lr_now:.2e} | TF {tf_ratio:.2f} | "
          f"Train {train_loss:.4f} | Val {val_loss:.4f} | "
          f"NSE {m['nse']:.3f} | MAE {m['mae']:.1f} | RMSE {m['rmse']:.1f} m3/s")
    history.append({"epoch": epoch+1, "lr": lr_now, "train_loss": train_loss, "val_loss": val_loss, **m})

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), ckpt_path)
        patience_cnt = 0
        print("  Best model saved")
    else:
        patience_cnt += 1
    if patience_cnt >= CFG.patience:
        print("Early stopping.")
        break


## Đánh giá Test (Holdout) + NSE theo lead-time + chẩn đoán từng trận lũ

In [ ]:
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for x_hind, x_nwp, y_b, station_rain, station_mask in test_loader:
        preds = model(x_hind.to(device), x_nwp.to(device), teacher_forcing_ratio=0.0,
                      station_rain=station_rain.to(device), station_mask=station_mask.to(device))
        all_preds.append(preds.cpu()); all_targets.append(y_b)

preds_cat, targets_cat = torch.cat(all_preds), torch.cat(all_targets)
m = compute_metrics(preds_cat, targets_cat, CFG.median_idx)
print(f"TEST — NSE={m['nse']:.4f}  MAE={m['mae']:.2f} m3/s  RMSE={m['rmse']:.2f} m3/s")

preds_np   = (preds_cat[:, :, CFG.median_idx] ** 2).numpy()
targets_np = (targets_cat ** 2).numpy()

horizon_rows = nse_per_horizon(preds_np, targets_np, group_hours=6)
for h in horizon_rows:
    print(f"  lead {h['hour_range']:>8}  NSE={h['nse']}")

obs_series, pred_series = extract_lead_time_series(preds_np, targets_np, lead_idx=CFG.forecast_len - 1)
event_diag = {"n_events": 0}
if len(obs_series) > 10 and obs_series.max() > 0:
    thr = float(np.percentile(obs_series, 90))
    event_diag = flood_event_diagnostics(obs_series, pred_series, threshold=thr)
    print(f"  [lead={CFG.forecast_len}h] n_events={event_diag['n_events']}  "
          f"NSE_event={event_diag.get('mean_event_nse')}  "
          f"peak_RE={event_diag.get('peak_re_mean')}  QA={event_diag.get('qa_pass_rate')}")

pd.DataFrame(history).to_excel(f"{OUTPUT_DIR}/lich_su_training.xlsx", index=False)
pd.DataFrame(horizon_rows).to_excel(f"{OUTPUT_DIR}/nse_theo_gio.xlsx", index=False)
with open(f"{OUTPUT_DIR}/metrics_test.json", "w", encoding="utf-8") as f:
    json.dump({"reservoir": RESERVOIR_NAME, "rid": RID, **m,
                "event_diagnostics": event_diag}, f, ensure_ascii=False, indent=2)

print(f"\nSaved to {OUTPUT_DIR}/: reservoir_lstm.pt, lich_su_training.xlsx, "
      f"nse_theo_gio.xlsx, metrics_test.json")
